# 02. QC: Bulk Methylation PCA and ELOVL2 Sanity Check

Performs cohort-wide quality control on bulk methylation beta files:

1. Discovers all bulk beta files across all 9 batches (no ML-QC filtering)
2. Randomly samples 50,000 CpG sites and builds a sample x site beta matrix
3. Applies missingness filters and computes PCA (PC1 + PC2 via randomized SVD)
4. Plots PCA colored by platform and sequencing source
5. Locates the ELOVL2 CpG index (chr6:11044644) and plots beta vs. age

**Inputs:** bulk `.beta` files from `results/<batch_token>/<pid>.beta`

**Outputs:** `results/pca_50k_sites_all_bulk_no_qc_with_v7/` on GCS

---

### Imports and configuration

In [ ]:
# ============================================================
# Bulk methylation PCA, all samples, no QC filtering, with v7
# ============================================================

import os
import io
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.auto import tqdm
from google.cloud import storage
from sklearn.decomposition import PCA

# ============================================================
# 0) CONFIG
# ============================================================

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
assert WORKSPACE_BUCKET.startswith("gs://")
BUCKET_NAME = WORKSPACE_BUCKET.replace("gs://", "")

client = storage.Client()

LOCAL_OUT_DIR = Path("/home/jupyter/pca_50k_sites_all_bulk_no_qc_with_v7")
LOCAL_OUT_DIR.mkdir(exist_ok=True)

GCS_OUT_DIR = f"{WORKSPACE_BUCKET}/results/pca_50k_sites_all_bulk_no_qc_with_v7"

# Total CpGs in hg38 beta representation used in your previous work
N_SITES_TOTAL = 29_152_891

# Randomly sampled CpGs for PCA
N_SAMPLE_SITES = 50_000
SITE_SEED = 20260426

# Beta files are your usual uint8 pair format: m, t
BETA_DTYPE = np.uint8

# Missingness filters after sampling
MIN_SAMPLE_OBS_FRAC = 0.50
MIN_SITE_OBS_FRAC = 0.80

print("WORKSPACE_BUCKET:", WORKSPACE_BUCKET)
print("LOCAL_OUT_DIR   :", LOCAL_OUT_DIR)
print("GCS_OUT_DIR     :", GCS_OUT_DIR)

### Plot style: colors, markers, matplotlib settings

In [ ]:
# ============================================================
# 1) STYLE AND COLORS
# ============================================================

COLORS = {
    "ONT":       "#185FA5",
    "Revio":     "#0F6E56",
    "Sequel2e":  "#993C1D",
    "V7_mixed":  "#6B6B6B",
}

SOURCE_ORDER = [
    "JHU_ONT",
    "BCM_ONT",

    "BI_Sequel2e",
    "UW_Sequel2e",
    "BCM_Sequel2e",

    "BI_Revio",
    "BCM_Revio",
    "UW_Revio",
    "HA_Revio",

    "V7_Base",
]

SOURCE_LABELS = {
    "JHU_ONT":       "JHU  ·  ONT",
    "BCM_ONT":       "BCM  ·  ONT",

    "BI_Sequel2e":   "Broad  ·  Sequel2e",
    "UW_Sequel2e":   "UW  ·  Sequel2e",
    "BCM_Sequel2e":  "BCM  ·  Sequel2e",

    "BI_Revio":   "Broad  ·  Revio",
    "BCM_Revio":     "BCM  ·  Revio",
    "UW_Revio":      "UW  ·  Revio",
    "HA_Revio":      "HA  ·  Revio",

    "V7_Base":       "V7 base  ·  mixed PacBio",
}

platform_order = ["ONT", "Revio", "Sequel2e", "V7_mixed"]

MARKERS = {
    "JHU_ONT": "o",
    "BCM_ONT": "s",

    "BI_Sequel2e": "^",
    "UW_Sequel2e": "v",
    "BCM_Sequel2e": "P",

    "BI_Revio": "o",
    "BCM_Revio": "s",
    "UW_Revio": "^",
    "HA_Revio": "D",

    "V7_Base": "X",
}

plt.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":          9,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.6,
    "xtick.major.width":  0.6,
    "ytick.major.width":  0.6,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "pdf.fonttype":       42,
    "ps.fonttype":        42,
})

### GCS helpers

In [ ]:
# ============================================================
# 2) GCS HELPERS
# ============================================================

def gs_to_bucket_blob(gs_path: str):
    assert gs_path.startswith("gs://"), gs_path
    rest = gs_path[5:]
    bkt, obj = rest.split("/", 1)
    return bkt, obj


def gcs_exists(gs_path: str) -> bool:
    bkt, obj = gs_to_bucket_blob(gs_path)
    return client.bucket(bkt).blob(obj).exists()


def upload_local_to_gcs(local_path: str, gcs_dir: str):
    local_path = str(local_path)
    cmd = f"gsutil cp {local_path} {gcs_dir}/{os.path.basename(local_path)}"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = r.returncode == 0
    print(("OK" if ok else "FAIL"), f"{gcs_dir}/{os.path.basename(local_path)}")
    if not ok:
        print(r.stderr[:1000])
    return ok


def load_beta_pairs_from_gcs(gs_path: str, dtype=np.uint8, min_bytes=1000) -> np.ndarray:
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    data = blob.download_as_bytes()

    if data is None or len(data) < min_bytes:
        raise ValueError(f"Empty/tiny beta file: {gs_path}")

    arr = np.frombuffer(data, dtype=dtype)

    if arr.size % 2:
        arr = arr[:-1]

    pairs = arr.reshape(-1, 2)

    if pairs.shape[0] == 0:
        raise ValueError(f"Parsed 0 CpGs from beta: {gs_path}")

    return pairs

### Discover all bulk beta files across all 9 batches (no QC filtering)

In [ ]:
# ============================================================
# 3) DISCOVER ALL BULK BETA FILES, NO QC FILTERING
# ============================================================

BULK_BATCHES = {
    # -------------------------
    # ONT
    # -------------------------
    "bcm_ont": {
        "raw_source": "BCM_ONT",
        "platform_group": "ONT",
    },
    "jhu_ont": {
        "raw_source": "JHU_ONT",
        "platform_group": "ONT",
    },

    # -------------------------
    # Sequel2e / older PacBio-style outputs
    # -------------------------
    "bi_sequel": {
        "raw_source": "BI_Sequel2e",
        "platform_group": "Sequel2e",
    },
    "bcm_sequel": {
        "raw_source": "BCM_Sequel2e",
        "platform_group": "Sequel2e",
    },
    "uw_sequel": {
        "raw_source": "UW_Sequel2e",
        "platform_group": "Sequel2e",
    },

    # -------------------------
    # Revio
    # -------------------------
    "bi_revio": {
        "raw_source": "BI_Revio",
        "platform_group": "Revio",
    },
    "bcm_revio": {
        "raw_source": "BCM_Revio",
        "platform_group": "Revio",
    },
    "uw_revio": {
        "raw_source": "UW_Revio",
        "platform_group": "Revio",
    },
    "ha_revio": {
        "raw_source": "HA_Revio",
        "platform_group": "Revio",
    },

    # -------------------------
    # v7 old mixed batch
    # -------------------------
    "v7_base": {
        "raw_source": "V7_Base",
        "platform_group": "V7_omited",
    },
}


def list_bulk_beta_files_for_batch(batch_token, raw_source, platform_group):
    """
    List beta files under WORKSPACE_BUCKET/results/<batch_token>/.
    This does not use ML-QC filtering.
    """
    prefix = f"results/{batch_token}/"
    rows = []

    print(f"Scanning {batch_token}: gs://{BUCKET_NAME}/{prefix}")

    for blob in client.list_blobs(BUCKET_NAME, prefix=prefix):
        if not blob.name.endswith(".beta"):
            continue

        fname = blob.name.split("/")[-1]
        person_id = fname.replace(".beta", "")

        # Expected bulk beta format: <person_id>.beta
        if not person_id.isdigit():
            continue

        size_bytes = int(blob.size or 0)

        rows.append({
            "person_id": str(person_id),
            "batch_token": batch_token,
            "raw_source": raw_source,
            "platform_group": platform_group,
            "bulk_beta_path": f"gs://{BUCKET_NAME}/{blob.name}",
            "size_bytes": size_bytes,
            "sample_key": f"{raw_source}__{person_id}",
        })

    return pd.DataFrame(rows)


manifest_parts = []

for batch_token, info in BULK_BATCHES.items():
    df = list_bulk_beta_files_for_batch(
        batch_token=batch_token,
        raw_source=info["raw_source"],
        platform_group=info["platform_group"],
    )
    manifest_parts.append(df)

all_bulk_manifest = (
    pd.concat(manifest_parts, ignore_index=True)
    .drop_duplicates(["sample_key", "bulk_beta_path"])
    .reset_index(drop=True)
)

# Only remove truly empty/tiny failed outputs, not ML-QC failures.
all_bulk_manifest = all_bulk_manifest[all_bulk_manifest["size_bytes"] >= 1000].copy()

print("All bulk beta manifest rows:", len(all_bulk_manifest))
print("Unique people:", all_bulk_manifest["person_id"].nunique())

display(
    all_bulk_manifest
    .groupby(["platform_group", "raw_source", "batch_token"])
    .size()
    .reset_index(name="n")
    .sort_values(["platform_group", "raw_source"])
)

pca_manifest = all_bulk_manifest.copy().reset_index(drop=True)

manifest_csv = LOCAL_OUT_DIR / "pca_input_manifest_all_bulk_no_qc_with_v7.csv"
pca_manifest.to_csv(manifest_csv, index=False)
upload_local_to_gcs(manifest_csv, GCS_OUT_DIR)

### Optional: merge ML-QC metrics for annotation (does not filter samples)

In [ ]:
# ============================================================
# 4) OPTIONAL: MERGE ML-QC METRICS, BUT DO NOT FILTER
# ============================================================

# This is only for annotation later.
# V7 will likely have missing QC metrics unless you ran ML-QC on v7 BAMs too.

QC_FULL_GS = (
    f"{WORKSPACE_BUCKET}/results/qc/bam_ml_qc_selfcontained/"
    "full_ml_qc_50k_all_platforms.csv"
)

def download_text_from_gcs(gs_path: str):
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    if not blob.exists():
        return None
    return blob.download_as_text()

def read_csv_from_gcs(gs_path: str) -> pd.DataFrame:
    txt = download_text_from_gcs(gs_path)
    if txt is None:
        raise FileNotFoundError(gs_path)
    return pd.read_csv(io.StringIO(txt))

try:
    full_qc_df = read_csv_from_gcs(QC_FULL_GS)

    full_qc_df["person_id"] = full_qc_df["person_id"].astype(str)
    full_qc_df["raw_source"] = full_qc_df["raw_source"].astype(str)
    full_qc_df["sample_key"] = (
        full_qc_df["raw_source"].astype(str) + "__" +
        full_qc_df["person_id"].astype(str)
    )

    qc_cols = [
        "sample_key",
        "ambiguous_fraction",
        "frac_reads_with_ML",
        "mean_ml_entropy",
        "total_ml_calls",
        "qc_status_raw",
    ]
    qc_cols = [c for c in qc_cols if c in full_qc_df.columns]

    pca_manifest = pca_manifest.merge(
        full_qc_df[qc_cols],
        on="sample_key",
        how="left",
    )

    print("Merged QC metrics.")
    print("Missing QC rows:", pca_manifest["ambiguous_fraction"].isna().sum())

except Exception as e:
    print("Could not merge QC metrics. Continuing without QC annotation.")
    print(type(e).__name__, e)

### Sample 50,000 CpG sites for PCA

In [ ]:
# ============================================================
# 5) SAMPLE 50,000 CpG SITES
# ============================================================

rng = np.random.default_rng(SITE_SEED)

site_idx = np.sort(
    rng.choice(N_SITES_TOTAL, size=N_SAMPLE_SITES, replace=False)
).astype(np.int32)

site_idx_path = LOCAL_OUT_DIR / f"sampled_{N_SAMPLE_SITES}_cpg_indices_seed{SITE_SEED}.npy"
np.save(site_idx_path, site_idx)

print("First sites:", site_idx[:10])
print("Last sites :", site_idx[-10:])
print("n sites    :", len(site_idx))

upload_local_to_gcs(site_idx_path, GCS_OUT_DIR)

### Build sample x CpG beta matrix

In [ ]:
# ============================================================
# 6) BUILD SAMPLE × CPG MATRIX
# ============================================================

def extract_beta_values_at_sites(gs_beta_path: str, site_idx: np.ndarray, dtype=np.uint8):
    pairs = load_beta_pairs_from_gcs(gs_beta_path, dtype=dtype)

    if pairs.shape[0] <= int(site_idx.max()):
        raise ValueError(
            f"Beta file too short: n_sites={pairs.shape[0]}, max_site={int(site_idx.max())}, path={gs_beta_path}"
        )

    m = pairs[site_idx, 0].astype(np.float32)
    t = pairs[site_idx, 1].astype(np.float32)

    beta = np.divide(
        m,
        t,
        out=np.full_like(m, np.nan, dtype=np.float32),
        where=t > 0,
    )

    coverage_fraction = float(np.mean(t > 0))
    mean_depth = float(np.nanmean(t)) if len(t) > 0 else np.nan

    return beta.astype(np.float32), coverage_fraction, mean_depth


n_samples = len(pca_manifest)
n_sites = len(site_idx)

X = np.full((n_samples, n_sites), np.nan, dtype=np.float32)

rows = []

for i, row in tqdm(
    enumerate(pca_manifest.itertuples(index=False)),
    total=n_samples,
    desc="Load sampled beta matrix"
):
    try:
        vals, cov_frac, mean_depth = extract_beta_values_at_sites(
            row.bulk_beta_path,
            site_idx,
            dtype=BETA_DTYPE,
        )

        X[i, :] = vals

        out = {
            "sample_key": row.sample_key,
            "person_id": row.person_id,
            "raw_source": row.raw_source,
            "platform_group": row.platform_group,
            "batch_token": row.batch_token,
            "bulk_beta_path": row.bulk_beta_path,
            "size_bytes": row.size_bytes,
            "coverage_fraction_50k": cov_frac,
            "mean_depth_50k": mean_depth,
            "matrix_status": "ok",
        }

        # Add optional QC annotation if present
        for c in ["ambiguous_fraction", "frac_reads_with_ML", "mean_ml_entropy", "total_ml_calls", "qc_status_raw"]:
            if hasattr(row, c):
                out[c] = getattr(row, c)

        rows.append(out)

    except Exception as e:
        rows.append({
            "sample_key": row.sample_key,
            "person_id": row.person_id,
            "raw_source": row.raw_source,
            "platform_group": row.platform_group,
            "batch_token": row.batch_token,
            "bulk_beta_path": row.bulk_beta_path,
            "size_bytes": row.size_bytes,
            "coverage_fraction_50k": np.nan,
            "mean_depth_50k": np.nan,
            "matrix_status": f"error:{type(e).__name__}:{str(e)[:120]}",
        })

matrix_status_df = pd.DataFrame(rows)

display(matrix_status_df["matrix_status"].value_counts(dropna=False))

matrix_status_csv = LOCAL_OUT_DIR / "pca_matrix_status_all_bulk_no_qc_with_v7.csv"
matrix_status_df.to_csv(matrix_status_csv, index=False)
upload_local_to_gcs(matrix_status_csv, GCS_OUT_DIR)

### Filter failed samples, apply missingness filter, impute site means

In [ ]:
# ============================================================
# 7) FILTER FAILED MATRIX ROWS, THEN MISSINGNESS FILTER
# ============================================================

ok_mask = matrix_status_df["matrix_status"].eq("ok").values

X_ok = X[ok_mask, :]
meta_ok = matrix_status_df[ok_mask].copy().reset_index(drop=True)

print("X_ok shape:", X_ok.shape)

display(
    meta_ok
    .groupby(["platform_group", "raw_source"])
    .size()
    .reset_index(name="n")
    .sort_values(["platform_group", "raw_source"])
)

# Sample missingness filter
sample_obs_frac = np.mean(~np.isnan(X_ok), axis=1)
keep_samples = sample_obs_frac >= MIN_SAMPLE_OBS_FRAC

X2 = X_ok[keep_samples, :]
meta2 = meta_ok.loc[keep_samples].copy().reset_index(drop=True)
meta2["sample_obs_frac_50k"] = sample_obs_frac[keep_samples]

# Site missingness filter
site_obs_frac = np.mean(~np.isnan(X2), axis=0)
keep_sites = site_obs_frac >= MIN_SITE_OBS_FRAC

X3 = X2[:, keep_sites]
site_idx_kept = site_idx[keep_sites]

print("Original X_ok shape        :", X_ok.shape)
print("After sample filter shape  :", X2.shape)
print("After site filter shape    :", X3.shape)
print("Kept site fraction         :", keep_sites.mean())
print("Kept sites                 :", len(site_idx_kept))

# Impute missing values by site mean
col_means = np.nanmean(X3, axis=0).astype(np.float32)

inds = np.where(np.isnan(X3))
X3[inds] = np.take(col_means, inds[1])

# Center columns, no variance scaling by default
X_centered = (X3 - col_means).astype(np.float32)

print("Any NaN after imputation:", np.isnan(X_centered).any())

meta_csv = LOCAL_OUT_DIR / "pca_samples_used_metadata_all_bulk_no_qc_with_v7.csv"
meta2.to_csv(meta_csv, index=False)
upload_local_to_gcs(meta_csv, GCS_OUT_DIR)

kept_sites_path = LOCAL_OUT_DIR / f"pca_kept_cpg_indices_from_{N_SAMPLE_SITES}_seed{SITE_SEED}.npy"
np.save(kept_sites_path, site_idx_kept)
upload_local_to_gcs(kept_sites_path, GCS_OUT_DIR)

### Compute PCA (2 components, randomized SVD)

In [ ]:
# ============================================================
# 8) FAST PCA, ONLY PC1 AND PC2
# ============================================================

pca = PCA(
    n_components=2,
    svd_solver="randomized",
    random_state=SITE_SEED,
)

scores = pca.fit_transform(X_centered)

meta2["PC1"] = scores[:, 0]
meta2["PC2"] = scores[:, 1]
meta2["PC1_var_explained"] = float(pca.explained_variance_ratio_[0])
meta2["PC2_var_explained"] = float(pca.explained_variance_ratio_[1])

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total PC1 + PC2:", pca.explained_variance_ratio_.sum())

scores_csv = LOCAL_OUT_DIR / "pca_scores_pc1_pc2_50k_sites_all_bulk_no_qc_with_v7.csv"
meta2.to_csv(scores_csv, index=False)
upload_local_to_gcs(scores_csv, GCS_OUT_DIR)

### Plot PC1 vs PC2 by platform

In [ ]:
# ============================================================
# 9) PLOT A: PC1 VS PC2 BY PLATFORM
# ============================================================

plt.figure(figsize=(6.4, 5.2), facecolor="white")

for p in platform_order:
    sub = meta2[meta2["platform_group"] == p]
    if len(sub) == 0:
        continue

    plt.scatter(
        sub["PC1"],
        sub["PC2"],
        s=11,
        alpha=0.55,
        color=COLORS[p],
        linewidths=0,
        rasterized=True,
        label=f"{p}  (n = {len(sub):,})",
    )

plt.xlabel(f"PC1 ({100 * pca.explained_variance_ratio_[0]:.2f}% variance)")
plt.ylabel(f"PC2 ({100 * pca.explained_variance_ratio_[1]:.2f}% variance)")
plt.title(
    f"PCA of bulk methylation beta values\n"
    f"All bulk samples, no ML-QC filtering, with v7, {len(site_idx_kept):,} CpGs",
    fontsize=10,
    fontweight="bold",
)
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()

pca_platform_png = LOCAL_OUT_DIR / "pca_pc1_pc2_by_platform_50k_all_bulk_no_qc_with_v7.png"
pca_platform_pdf = LOCAL_OUT_DIR / "pca_pc1_pc2_by_platform_50k_all_bulk_no_qc_with_v7.pdf"

plt.savefig(pca_platform_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(pca_platform_pdf, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

upload_local_to_gcs(pca_platform_png, GCS_OUT_DIR)
upload_local_to_gcs(pca_platform_pdf, GCS_OUT_DIR)

### Plot PC1 vs PC2 by sequencing source

In [ ]:
# ============================================================
# 10) PLOT B: PC1 VS PC2 BY SOURCE
# ============================================================

plt.figure(figsize=(7.6, 5.8), facecolor="white")

present_sources = [s for s in SOURCE_ORDER if s in meta2["raw_source"].unique()]

for src in present_sources:
    sub = meta2[meta2["raw_source"] == src]
    if len(sub) == 0:
        continue

    p = sub["platform_group"].iloc[0]

    plt.scatter(
        sub["PC1"],
        sub["PC2"],
        s=14,
        alpha=0.55,
        color=COLORS.get(p, "#777777"),
        marker=MARKERS.get(src, "o"),
        linewidths=0,
        rasterized=True,
        label=f"{SOURCE_LABELS.get(src, src)}  (n = {len(sub):,})",
    )

plt.xlabel(f"PC1 ({100 * pca.explained_variance_ratio_[0]:.2f}% variance)")
plt.ylabel(f"PC2 ({100 * pca.explained_variance_ratio_[1]:.2f}% variance)")
plt.title(
    f"PCA by sequencing source\n"
    f"All bulk samples, no ML-QC filtering, with v7, {len(site_idx_kept):,} CpGs",
    fontsize=10,
    fontweight="bold",
)
plt.legend(frameon=False, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

pca_source_png = LOCAL_OUT_DIR / "pca_pc1_pc2_by_source_50k_all_bulk_no_qc_with_v7.png"
pca_source_pdf = LOCAL_OUT_DIR / "pca_pc1_pc2_by_source_50k_all_bulk_no_qc_with_v7.pdf"

plt.savefig(pca_source_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(pca_source_pdf, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

upload_local_to_gcs(pca_source_png, GCS_OUT_DIR)
upload_local_to_gcs(pca_source_pdf, GCS_OUT_DIR)

### Combined figure: platform + source panels

In [ ]:
# ============================================================
# 11) COMBINED FIGURE: PLATFORM AND SOURCE
# ============================================================

fig = plt.figure(figsize=(12.2, 5.3), facecolor="white")
gs = gridspec.GridSpec(
    1,
    2,
    figure=fig,
    width_ratios=[1, 1.25],
    wspace=0.36,
)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

# Panel A: platform
for p in platform_order:
    sub = meta2[meta2["platform_group"] == p]
    if len(sub) == 0:
        continue

    ax1.scatter(
        sub["PC1"],
        sub["PC2"],
        s=11,
        alpha=0.55,
        color=COLORS[p],
        linewidths=0,
        rasterized=True,
        label=f"{p}  (n = {len(sub):,})",
    )

ax1.set_xlabel(f"PC1 ({100 * pca.explained_variance_ratio_[0]:.2f}%)")
ax1.set_ylabel(f"PC2 ({100 * pca.explained_variance_ratio_[1]:.2f}%)")
ax1.set_title("A     PCA by platform", fontsize=9, fontweight="bold", loc="left", pad=6)
ax1.legend(frameon=False, fontsize=8, loc="best")
ax1.tick_params(labelsize=8)

# Panel B: source
for src in present_sources:
    sub = meta2[meta2["raw_source"] == src]
    if len(sub) == 0:
        continue

    p = sub["platform_group"].iloc[0]

    ax2.scatter(
        sub["PC1"],
        sub["PC2"],
        s=13,
        alpha=0.55,
        color=COLORS.get(p, "#777777"),
        marker=MARKERS.get(src, "o"),
        linewidths=0,
        rasterized=True,
        label=f"{SOURCE_LABELS.get(src, src)}  (n = {len(sub):,})",
    )

ax2.set_xlabel(f"PC1 ({100 * pca.explained_variance_ratio_[0]:.2f}%)")
ax2.set_ylabel(f"PC2 ({100 * pca.explained_variance_ratio_[1]:.2f}%)")
ax2.set_title("B     PCA by source", fontsize=9, fontweight="bold", loc="left", pad=6)
ax2.legend(frameon=False, fontsize=6.8, bbox_to_anchor=(1.02, 1), loc="upper left")
ax2.tick_params(labelsize=8)

fig.suptitle(
    f"Bulk methylation PCA using {len(site_idx_kept):,} sampled CpGs",
    fontsize=11,
    fontweight="bold",
    y=1.02,
)

caption = (
    "PCA was computed on bulk beta values from all available preprocessed long-read methylation outputs, "
    "without ML-QC filtering. v7 samples were included as a separate mixed PacBio group. "
    f"CpGs were randomly sampled, sites with observation fraction below {MIN_SITE_OBS_FRAC:.0%} were excluded, "
    "and missing beta values were imputed with site means. Only PC1 and PC2 were computed using randomized PCA."
)

fig.text(
    0.5,
    -0.055,
    caption,
    ha="center",
    fontsize=7.5,
    color="#5F5E5A",
    style="italic",
    wrap=True,
)

combined_png = LOCAL_OUT_DIR / "fig_pca_pc1_pc2_50k_all_bulk_no_qc_with_v7_combined.png"
combined_pdf = LOCAL_OUT_DIR / "fig_pca_pc1_pc2_50k_all_bulk_no_qc_with_v7_combined.pdf"

plt.savefig(combined_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(combined_pdf, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

upload_local_to_gcs(combined_png, GCS_OUT_DIR)
upload_local_to_gcs(combined_pdf, GCS_OUT_DIR)

### Save PCA summary CSV

In [ ]:
# ============================================================
# 12) SAVE PCA SUMMARY
# ============================================================

summary = pd.DataFrame([{
    "analysis": "bulk_pca_all_samples_no_qc_with_v7",
    "n_samples_manifest": int(len(pca_manifest)),
    "n_samples_matrix_ok": int(len(meta_ok)),
    "n_samples_used_pca": int(len(meta2)),
    "n_sites_sampled_initial": int(N_SAMPLE_SITES),
    "n_sites_used_after_filter": int(len(site_idx_kept)),
    "site_seed": int(SITE_SEED),
    "min_sample_obs_frac": float(MIN_SAMPLE_OBS_FRAC),
    "min_site_obs_frac": float(MIN_SITE_OBS_FRAC),
    "pc1_var_explained": float(pca.explained_variance_ratio_[0]),
    "pc2_var_explained": float(pca.explained_variance_ratio_[1]),
    "pc1_pc2_var_explained_total": float(pca.explained_variance_ratio_.sum()),
}])

summary_csv = LOCAL_OUT_DIR / "pca_50k_all_bulk_no_qc_with_v7_summary.csv"
summary.to_csv(summary_csv, index=False)
upload_local_to_gcs(summary_csv, GCS_OUT_DIR)

display(summary)

print("\nDone.")
print("Local outputs:", LOCAL_OUT_DIR)
print("GCS outputs  :", GCS_OUT_DIR)

---
## ELOVL2 Sanity Check

Locate the exact wgbstools CpG index for chr6:11044644 (ELOVL2 clock locus), then plot bulk beta values against age to confirm the known positive correlation. This verifies that the methylation pipeline is working correctly end-to-end.

**Prerequisite:** `CPG_BED_GZ` must point to the wgbstools `CpG.bed.gz` hg38 annotation file.

In [ ]:
# Path to wgbstools CpG.bed.gz (hg38 reference)
# Adjust to match where wgbstools stores its genome reference on your workbench
import os
WGBS_TOOLS_DIR = os.path.join(os.getcwd(), 'wgbs_tools')
CPG_BED_GZ = os.path.join(WGBS_TOOLS_DIR, 'references', 'hg38', 'CpG.bed.gz')

import os
if not os.path.exists(CPG_BED_GZ):
    # Fallback: try to find it in the WGBSTOOLS installation
    import subprocess
    result = subprocess.run(
        ['find', WGBS_TOOLS_DIR, '-name', 'CpG.bed.gz'],
        capture_output=True, text=True
    )
    candidates = result.stdout.strip().split('\n')
    if candidates and candidates[0]:
        CPG_BED_GZ = candidates[0]
        print('Found CPG_BED_GZ:', CPG_BED_GZ)
    else:
        print('WARNING: CPG_BED_GZ not found. Set manually before running ELOVL2 cells.')
else:
    print('CPG_BED_GZ:', CPG_BED_GZ)


### Find exact wgbstools CpG index for chr6:11044644 (ELOVL2)

In [ ]:
# ============================================================
# Find exact wgbstools CpG index for chr6:11044644
# ============================================================

import gzip
import pandas as pd
import numpy as np

TARGET_CHROM = "chr6"
TARGET_POS_1BASED = 11044644

print("Using CPG_BED_GZ:", CPG_BED_GZ)
print("Target:", TARGET_CHROM, TARGET_POS_1BASED)

def find_exact_cpg_index_wgbstools(cpg_bed_gz, chrom, target_pos_1based, flank=5):
    """
    Find the CpG index in wgbstools CpG.bed.gz corresponding to a 1-based hg38 coordinate.

    We inspect both:
      start0 + 1 == target_pos_1based
      end0 == target_pos_1based
      start0 == target_pos_1based

    because CpG.bed conventions can be easy to misread.
    The beta row index should correspond to the 0-based line number in CpG.bed.gz.
    """
    hits = []
    nearby = []

    with gzip.open(cpg_bed_gz, "rt") as f:
        for idx0, line in enumerate(f):
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue

            c = parts[0]
            if c != chrom:
                continue

            start0 = int(parts[1])
            end0 = int(parts[2])

            pos_start_plus1 = start0 + 1
            pos_end = end0
            pos_start0 = start0

            # Save nearby rows for diagnostics
            if abs(pos_start_plus1 - target_pos_1based) <= flank or abs(pos_end - target_pos_1based) <= flank:
                nearby.append({
                    "cpg_index": idx0,
                    "chrom": c,
                    "start0": start0,
                    "end0": end0,
                    "start0_plus1": pos_start_plus1,
                    "end0_as_1based": pos_end,
                    "distance_start0_plus1": abs(pos_start_plus1 - target_pos_1based),
                    "distance_end0": abs(pos_end - target_pos_1based),
                    "raw_line": line.rstrip("\n"),
                })

            # Exact matching rules
            if pos_start_plus1 == target_pos_1based:
                hits.append({
                    "match_rule": "start0_plus1_equals_target",
                    "cpg_index": idx0,
                    "chrom": c,
                    "start0": start0,
                    "end0": end0,
                    "start0_plus1": pos_start_plus1,
                    "end0_as_1based": pos_end,
                    "raw_line": line.rstrip("\n"),
                })

            if pos_end == target_pos_1based:
                hits.append({
                    "match_rule": "end0_equals_target",
                    "cpg_index": idx0,
                    "chrom": c,
                    "start0": start0,
                    "end0": end0,
                    "start0_plus1": pos_start_plus1,
                    "end0_as_1based": pos_end,
                    "raw_line": line.rstrip("\n"),
                })

            if pos_start0 == target_pos_1based:
                hits.append({
                    "match_rule": "start0_equals_target",
                    "cpg_index": idx0,
                    "chrom": c,
                    "start0": start0,
                    "end0": end0,
                    "start0_plus1": pos_start_plus1,
                    "end0_as_1based": pos_end,
                    "raw_line": line.rstrip("\n"),
                })

    hits_df = pd.DataFrame(hits).drop_duplicates()
    nearby_df = pd.DataFrame(nearby).drop_duplicates()

    return hits_df, nearby_df


target_hits_df, target_nearby_df = find_exact_cpg_index_wgbstools(
    CPG_BED_GZ,
    TARGET_CHROM,
    TARGET_POS_1BASED,
    flank=10,
)

print("Exact hits:")
display(target_hits_df)

print("Nearby CpGs:")
display(target_nearby_df.sort_values(["distance_start0_plus1", "distance_end0"]).head(20))

### Select the target CpG index

In [ ]:
# ============================================================
# Choose the exact target CpG index
# ============================================================

if len(target_hits_df) == 0:
    raise ValueError(
        f"No exact CpG found for {TARGET_CHROM}:{TARGET_POS_1BASED}. "
        "Inspect target_nearby_df to confirm coordinate convention."
    )

preferred = target_hits_df[target_hits_df["match_rule"] == "start0_plus1_equals_target"].copy()

if len(preferred) == 0:
    preferred = target_hits_df[target_hits_df["match_rule"] == "end0_equals_target"].copy()

if len(preferred) == 0:
    preferred = target_hits_df.copy()

target_cpg_row = preferred.iloc[0]
TARGET_CPG_INDEX = int(target_cpg_row["cpg_index"])

print("Selected target CpG:")
display(pd.DataFrame([target_cpg_row]))

print("TARGET_CPG_INDEX:", TARGET_CPG_INDEX)

### Merge exact age at blood draw into pca_manifest (BigQuery + genomic_metrics.tsv)

In [ ]:
# ============================================================
# Merge exact age at blood draw into pca_manifest
# ============================================================

import os
import pandas as pd

if "pca_manifest" not in globals():
    raise NameError("pca_manifest is not defined. Run the all-bulk manifest cell first.")

# -----------------------------
# 1) Query demographics
# -----------------------------
dataset_36069554_person_sql = """
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person
    LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept
        ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id
    WHERE
        person.PERSON_ID IN (
            SELECT DISTINCT person_id
            FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person`
            WHERE has_lr_whole_genome_variant = 1
        )
"""

print("Querying demographics from BigQuery...")
demo_df = pd.read_gbq(
    dataset_36069554_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook",
)

demo_df["person_id"] = demo_df["person_id"].astype(str)
print("Demographic rows:", len(demo_df))

# -----------------------------
# 2) Load biosample collection date
# -----------------------------
qc_metrics_path = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/genomic_metrics.tsv"

print(f"Loading QC metrics from: {qc_metrics_path}")
qc_df = pd.read_csv(
    qc_metrics_path,
    sep="\t",
    storage_options={
        "requester_pays": True,
        "project": os.environ["GOOGLE_PROJECT"],
    },
)

if "sample_name" in qc_df.columns:
    qc_df = qc_df.rename(columns={"sample_name": "person_id"})
elif "research_id" in qc_df.columns:
    qc_df = qc_df.rename(columns={"research_id": "person_id"})

qc_df["person_id"] = qc_df["person_id"].astype(str)

if "biosample_collection_date" not in qc_df.columns:
    raise ValueError("biosample_collection_date not found in genomic_metrics.tsv")

# -----------------------------
# 3) Calculate exact age at blood draw
# -----------------------------
clinical_df = demo_df.merge(
    qc_df[["person_id", "biosample_collection_date"]],
    on="person_id",
    how="inner",
).copy()

clinical_df["date_of_birth"] = pd.to_datetime(clinical_df["date_of_birth"], utc=True)
clinical_df["biosample_collection_date"] = pd.to_datetime(
    clinical_df["biosample_collection_date"],
    utc=True,
)

clinical_df["Age"] = (
    clinical_df["biosample_collection_date"] - clinical_df["date_of_birth"]
).dt.days / 365.25

clinical_df["Age"] = pd.to_numeric(clinical_df["Age"], errors="coerce")
clinical_df = clinical_df.dropna(subset=["Age"])
clinical_df = clinical_df[clinical_df["Age"] > 0].copy()

clinical_df = clinical_df[
    ["person_id", "Age", "sex_at_birth", "biosample_collection_date"]
].drop_duplicates(subset=["person_id"])

print("Clinical participants with usable age:", len(clinical_df))

# -----------------------------
# 4) Merge into pca_manifest
# -----------------------------
pca_manifest = pca_manifest.copy()
pca_manifest["person_id"] = pca_manifest["person_id"].astype(str)

# Drop stale columns if present
pca_manifest = pca_manifest.drop(
    columns=[
        c for c in ["Age", "sex_at_birth", "biosample_collection_date"]
        if c in pca_manifest.columns
    ],
    errors="ignore",
)

pca_manifest = pca_manifest.merge(
    clinical_df,
    on="person_id",
    how="left",
)

print("pca_manifest rows:", len(pca_manifest))
print("Rows with Age:", pca_manifest["Age"].notna().sum())
print("Rows missing Age:", pca_manifest["Age"].isna().sum())

display(
    pca_manifest.groupby(["platform_group", "raw_source"])
    .agg(
        n=("sample_key", "nunique"),
        n_age=("Age", lambda x: x.notna().sum()),
        age_min=("Age", "min"),
        age_max=("Age", "max"),
    )
    .reset_index()
    .sort_values(["platform_group", "raw_source"])
)

# Optional save
if "LOCAL_OUT_DIR" in globals():
    age_manifest_csv = LOCAL_OUT_DIR / "pca_input_manifest_all_bulk_no_qc_with_v7_with_exact_age.csv"
    pca_manifest.to_csv(age_manifest_csv, index=False)
    print("Saved local:", age_manifest_csv)

    if "upload_local_to_gcs" in globals() and "GCS_OUT_DIR" in globals():
        upload_local_to_gcs(age_manifest_csv, GCS_OUT_DIR)

### Extract ELOVL2 beta value per sample from bulk beta files

In [ ]:
# ============================================================
# Extract exact chr6:11044644 ELOVL2 CpG beta per sample
# ============================================================

from tqdm.auto import tqdm
from pathlib import Path
import os

if "pca_manifest" not in globals():
    raise NameError("pca_manifest is missing. Run the all-bulk manifest cell first.")

if "Age" not in pca_manifest.columns:
    raise NameError("Age is missing from pca_manifest. Merge exact blood-draw age first.")

if "BETA_DTYPE" not in globals():
    BETA_DTYPE = np.uint8

LOCAL_ELOVL2_DIR = Path("/home/jupyter/elovl2_sanity_all_bulk_no_qc_with_v7")
LOCAL_ELOVL2_DIR.mkdir(exist_ok=True)

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
GCS_ELOVL2_DIR = f"{WORKSPACE_BUCKET}/results/elovl2_sanity_all_bulk_no_qc_with_v7"

def extract_exact_cpg_beta(gs_beta_path, cpg_index, dtype=np.uint8):
    pairs = load_beta_pairs_from_gcs(gs_beta_path, dtype=dtype)

    if pairs.shape[0] <= cpg_index:
        raise ValueError(f"Beta file too short for cpg_index={cpg_index}: {gs_beta_path}")

    m = float(pairs[cpg_index, 0])
    t = float(pairs[cpg_index, 1])

    beta = m / t if t > 0 else np.nan

    return m, t, beta

rows = []

for row in tqdm(pca_manifest.itertuples(index=False), total=len(pca_manifest), desc="Extract exact ELOVL2 CpG"):
    try:
        m, t, beta = extract_exact_cpg_beta(
            row.bulk_beta_path,
            TARGET_CPG_INDEX,
            dtype=BETA_DTYPE,
        )
        status = "ok"

    except Exception as e:
        m, t, beta = np.nan, np.nan, np.nan
        status = f"error:{type(e).__name__}:{str(e)[:120]}"

    rows.append({
        "sample_key": getattr(row, "sample_key", None),
        "person_id": str(getattr(row, "person_id", "")),
        "raw_source": getattr(row, "raw_source", None),
        "platform_group": getattr(row, "platform_group", None),
        "batch_token": getattr(row, "batch_token", None),
        "bulk_beta_path": getattr(row, "bulk_beta_path", None),
        "Age": getattr(row, "Age", np.nan),

        "target_chrom": TARGET_CHROM,
        "target_pos_1based": TARGET_POS_1BASED,
        "target_cpg_index": TARGET_CPG_INDEX,
        "target_cpg_start0": int(target_cpg_row["start0"]),
        "target_cpg_end0": int(target_cpg_row["end0"]),
        "target_match_rule": target_cpg_row["match_rule"],

        "elovl2_exact_m": m,
        "elovl2_exact_t": t,
        "elovl2_exact_beta": beta,
        "elovl2_exact_status": status,
    })

elovl2_exact_df = pd.DataFrame(rows)

print("Exact ELOVL2 extraction status:")
display(elovl2_exact_df["elovl2_exact_status"].value_counts(dropna=False).rename("n").to_frame())

display(
    elovl2_exact_df.groupby(["platform_group", "raw_source"])
    .agg(
        n=("sample_key", "nunique"),
        n_age=("Age", lambda x: x.notna().sum()),
        n_ok=("elovl2_exact_status", lambda x: (x == "ok").sum()),
        median_depth=("elovl2_exact_t", "median"),
        median_beta=("elovl2_exact_beta", "median"),
    )
    .reset_index()
    .sort_values(["platform_group", "raw_source"])
)

exact_csv = LOCAL_ELOVL2_DIR / "elovl2_exact_chr6_11044644_beta_all_bulk_no_qc_with_v7.csv"
elovl2_exact_df.to_csv(exact_csv, index=False)
upload_local_to_gcs(exact_csv, GCS_ELOVL2_DIR)

print("Saved:", exact_csv)

### Regression plot: ELOVL2 beta vs. chronological age

In [ ]:
# ============================================================
# Regression plot: exact chr6:11044644 ELOVL2 CpG beta vs age
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

COLORS = {
    "ONT":       "#185FA5",
    "Revio":     "#0F6E56",
    "Sequel2e":  "#993C1D",
    "V7_mixed":  "#6B6B6B",
}

platform_order = ["ONT", "Revio", "Sequel2e", "V7_mixed"]

plot_df = elovl2_exact_df[
    (elovl2_exact_df["elovl2_exact_status"] == "ok") &
    (elovl2_exact_df["Age"].notna()) &
    (elovl2_exact_df["elovl2_exact_beta"].notna()) &
    (elovl2_exact_df["elovl2_exact_t"].fillna(0) > 0)
].copy()

print("Rows available for exact CpG plot:", len(plot_df))

fig, axes = plt.subplots(2, 2, figsize=(10, 8.2), facecolor="white")
axes = axes.flatten()

for ax, p in zip(axes, platform_order):
    sub = plot_df[plot_df["platform_group"] == p].copy()

    if len(sub) == 0:
        ax.axis("off")
        continue

    x = sub["Age"].values.astype(float)
    y = sub["elovl2_exact_beta"].values.astype(float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]

    ax.scatter(
        x,
        y,
        s=12,
        alpha=0.45,
        color=COLORS[p],
        linewidths=0,
        rasterized=True,
    )

    if len(x) >= 3:
        fit = stats.linregress(x, y)
        xs = np.linspace(np.min(x), np.max(x), 200)
        ys = fit.intercept + fit.slope * xs

        ax.plot(
            xs,
            ys,
            color=COLORS[p],
            linewidth=2.0,
            alpha=0.95,
        )

        txt = (
            f"n = {len(x)}\n"
            f"median depth = {sub['elovl2_exact_t'].median():.1f}\n"
            f"slope = {fit.slope:.4f} / yr\n"
            f"r = {fit.rvalue:.3f}\n"
            f"$R^2$ = {fit.rvalue**2:.3f}\n"
            f"p = {fit.pvalue:.2e}"
        )

        ax.text(
            0.03,
            0.97,
            txt,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            bbox=dict(
                boxstyle="round,pad=0.3",
                facecolor="white",
                edgecolor="#CCCCCC",
                alpha=0.9,
            ),
        )

    ax.set_title(p, fontsize=10, fontweight="bold")
    ax.set_xlabel("Chronological age at blood draw (years)")
    ax.set_ylabel("ELOVL2 beta at chr6:11044644")

fig.suptitle(
    f"Exact ELOVL2 CpG methylation vs age\n"
    f"{TARGET_CHROM}:{TARGET_POS_1BASED}, wgbstools CpG index {TARGET_CPG_INDEX}",
    fontsize=12,
    fontweight="bold",
    y=0.98,
)

fig.tight_layout(rect=[0, 0.02, 1, 0.94])

out_png = LOCAL_ELOVL2_DIR / "elovl2_exact_chr6_11044644_regression_by_platform.png"
out_pdf = LOCAL_ELOVL2_DIR / "elovl2_exact_chr6_11044644_regression_by_platform.pdf"

fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(out_pdf, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

upload_local_to_gcs(out_png, GCS_ELOVL2_DIR)
upload_local_to_gcs(out_pdf, GCS_ELOVL2_DIR)